In [ ]:
import ollama
import json

prompt_config_path = ""
host = ""
model = ""
available_error_types = ""

In [ ]:
# get the prompts from the json file

try:
    with open(prompt_config_path, "r", encoding="utf-8") as file:
        prompts = json.load(file)

        system_prompt_error = prompts['system_prompt_error']
        system_prompt_suggestion = prompts['system_prompt_suggestion']
        get_error_prompt = prompts['get_error_prompt']
        suggestion_prompt = prompts['suggestion_prompt']

        print(f"'{prompt_config_path}' successfully loaded!")
        
except FileNotFoundError:
    print(f"Error: The file '{prompt_config_path}' was not found.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")

client = ollama.Client(host=host)

In [ ]:
# get error prompt

def request_error(get_error_prompt: str) -> str:
    get_error_prompt = get_error_prompt.format(
        code = "", # the code in string format containing the changes in a specific pull request mined from the repo
        available_error_types = available_error_types
    )

    get_error_response = client.chat(
        model = model,
        messages = [
            {"role": "system", "content": system_prompt_error},
            {"role": "user", "content": get_error_prompt}
        ]
    )

    error_response = get_error_response["message"]["content"]
    return error_response

def parse_error_response(error_response: str) -> list[Error]:
    # will parse the response from llm
    # <ADD HERE>

    parsed_errors = [] # end result or parsing the error response from LLM: a list of error information in consistent format

    # Create an Error object. Assign error type, severity level, and error description generated from llm
    errors = []

    for idx, error_info in enumerate(parsed_errors):
        error_type = ""
        error_severity_level = -1
        error_description = ""

        error = Error(error_type, error_severity_level, error_description)
        errors.append(error)

    return errors

error_response = request_error(get_error_prompt)
errors = parse_error_response(error_response)

In [ ]:
# get fix suggestion

def request_suggestion(error: Error) -> Error:
    error_type = error.get_error_type()
    severity = error.get_error_severity_level()
    error_description = error.get_error_description()

    suggestion_prompt = suggestion_prompt.format(
        error_type = error_type,
        severity = severity,
        error_description = error_description
    )

    suggestion_response = client.chat(
        model = model,
        messages = [
            {"role": "system", "content": system_prompt_suggestion},
            {"role": "user", "content": suggestion_prompt}
        ]
    )

    suggestion_response = suggestion_response["message"]["content"]

    error.set_fix_suggestion(suggestion_response)

    return error_response

def get_all_fix_suggestions(errors: list[Error]) -> list[Error]:
    for idx, error in enumerate(errors):
        errors[idx] = request_suggestion(error)

    return errors

errors = get_all_fix_suggestions(errors)